# 18：ウツタインデータをTabDiffで学習し10,000例生成
`../data/processed/utstein_modified.pkl` → 前処理 → 公式TabDiff学習 → 10,000例生成 → PKL/CSV保存。

このファイルをプロジェクトの`notebooks`へ置き、上から実行してください。実患者の再抽出ではなく学習済み拡散モデルから生成します。生成データの臨床整合性・分布・プライバシーは別途評価が必要です。

公式実装：https://github.com/MinkaiXu/TabDiff （確認コミット：5ecdb3356261aea72716cc9a779f31d7ad083bf4）

## 環境（初回のみターミナルで実行）
Python 3.10環境を推奨。既存のPython 3.14環境と分けてください。公式環境定義は古いCUDA版なので、新しいGPUでは対応するPyTorch/CUDAを別途選んでください。
```bash
git clone https://github.com/MinkaiXu/TabDiff.git ../external/TabDiff
conda env create -f ../external/TabDiff/tabdiff.yaml
conda activate tabdiff
python -m pip install ipykernel
python -m ipykernel install --user --name tabdiff --display-name "Python (TabDiff)"
```
Jupyterでこのカーネルを選択します。Notebookは依存関係を自動インストールしません。

In [1]:
from pathlib import Path
from datetime import datetime
import os, sys, json, subprocess, hashlib
import numpy as np
import pandas as pd
from IPython.display import display

PROJECT = Path.cwd().resolve()
if PROJECT.name == "notebooks":
    PROJECT = PROJECT.parent
INPUT = PROJECT / "data/processed/utstein_modified.pkl"
TABDIFF_REPO = PROJECT / "external/TabDiff"  # 既存の公式コードがあれば変更
TABDIFF_PYTHON = Path(sys.executable)       # TabDiff環境のPython
N_GENERATE = 10_000
EPOCHS = 200                              # 動作確認なら2、本学習は損失を見て調整
TRAIN_BATCH_SIZE = 1024
SAMPLE_BATCH_SIZE = 512                   # GPUメモリ不足なら下げる
GPU = 0                                  # CPUなら-1（大規模データは非常に時間がかかります）
SEED = 42
MAX_TRAIN_ROWS = None                     # 全件使用。試運転のみ例：20000
EXCLUDE_COLUMNS = ["ID", "id", "患者ID", "患者番号", "症例ID"]
RUN_ID = datetime.now().strftime("%Y%m%d_%H%M%S_%f")
DATA_NAME = "utstein_" + RUN_ID
EXP_NAME = "unconditional"
OUT = PROJECT / "artifacts/18_tabdiff" / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
assert INPUT.is_file(), f"入力がありません: {INPUT}"
assert (TABDIFF_REPO / "main.py").is_file(), f"公式TabDiffのパスを指定してください: {TABDIFF_REPO}"
print("入力:", INPUT, "\n出力:", OUT)

入力: /Users/t_aokage/Projects/Utstein_Python/data/processed/utstein_modified.pkl 
出力: /Users/t_aokage/Projects/Utstein_Python/artifacts/18_tabdiff/20261020_170859_713239


In [2]:
# 環境確認。外部ログ送信は無効化します。
def run_command(args, log_name):
    env = os.environ.copy()
    env.update(WANDB_MODE="disabled", PYTHONUNBUFFERED="1", PYTHONHASHSEED=str(SEED))
    print("実行:", " ".join(map(str, args)))
    with (OUT / log_name).open("w", encoding="utf-8") as log:
        proc = subprocess.Popen(list(map(str, args)), cwd=TABDIFF_REPO, env=env,
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        try:
            for line in proc.stdout:
                print(line, end="")
                log.write(line)
            status = proc.wait()
        except KeyboardInterrupt:
            proc.terminate()
            proc.wait()
            raise
    if status:
        raise RuntimeError(f"処理失敗。ログを確認: {OUT / log_name}")

run_command([TABDIFF_PYTHON, "-c",
    "import torch,pandas,numpy,sklearn,sdmetrics,wandb,tomli,tomli_w; "
    "print('torch',torch.__version__);print('CUDA',torch.cuda.is_available()); "
    "print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')"], "environment.log")
commit = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=TABDIFF_REPO, text=True).strip()
print("TabDiff commit:", commit)

実行: /Users/t_aokage/Projects/Utstein_Python/.venv/bin/python -c import torch,pandas,numpy,sklearn,sdmetrics,wandb,tomli,tomli_w; print('torch',torch.__version__);print('CUDA',torch.cuda.is_available()); print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
torch 2.14.1
CUDA False
CPU
TabDiff commit: 5ecdb3356261aea72716cc9a779f31d7ad083bf4


## データの前処理
全行をランダムに学習90%・ホールドアウト10%へ分割します。数値の中央値は学習側のみから計算します。
数値欠損は中央値で埋め、欠損フラグを追加して欠損パターンも学習します。カテゴリは内部文字コードに変換し、生成後に復元します。日時は1970年からの経過日数として数値化します。IDは学習しません。
全欠損列・定数列はモデルから外し、出力時に復元します。カテゴリが非常に多い列はエラーにして確認を求めます。

In [3]:
real = pd.read_pickle(INPUT)
if not isinstance(real, pd.DataFrame) or not real.columns.is_unique:
    raise ValueError("入力は列名が一意のDataFrameにしてください。")
real = real.drop(columns=[c for c in EXCLUDE_COLUMNS if c in real.columns]).copy()
if MAX_TRAIN_ROWS is not None:
    real = real.sample(n=min(MAX_TRAIN_ROWS, len(real)), random_state=SEED)
real = real.reset_index(drop=True)
if len(real) < 20:
    raise ValueError("少なくとも20行必要です。")
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(real))
cut = int(len(real) * .9)
train_idx, test_idx = indices[:cut], indices[cut:]
schema, encoded = {}, pd.DataFrame(index=real.index)
constants = {}
num_names, cat_names = [], []
for i, column in enumerate(real.columns):
    series = real[column]
    key = f"f{i:03d}"
    # 日付のみのobject列にも対応
    is_date = pd.api.types.is_datetime64_any_dtype(series) or column in ["発生年月日ウツタイン", "覚知ウツタイン"]
    if series.nunique(dropna=False) <= 1:
        constants[column] = series.iloc[0]
        continue
    spec = {"key": key, "dtype": series.dtype}
    if is_date or (pd.api.types.is_numeric_dtype(series) and not isinstance(series.dtype, pd.CategoricalDtype)):
        if is_date:
            dt = pd.to_datetime(series, errors="coerce")
            values = (dt - pd.Timestamp("1970-01-01")).dt.total_seconds() / 86400
            spec["kind"] = "datetime"
        else:
            values = pd.to_numeric(series, errors="coerce").astype(float)
            spec["kind"] = "numeric"
        values = values.replace([np.inf, -np.inf], np.nan)
        observed = values.loc[train_idx].dropna()
        if observed.empty:
            raise ValueError(f"学習側に有効な数値がありません: {column}")
        spec.update(median=float(observed.median()), minimum=float(observed.min()),
                    maximum=float(observed.max()), integer=bool((observed % 1 == 0).all()))
        encoded[key] = values.fillna(spec["median"])
        num_names.append(key)
        if values.isna().any():
            spec["mask"] = key + "_missing"
            encoded[spec["mask"]] = np.where(values.isna(), "missing_yes", "missing_no")
            cat_names.append(spec["mask"])
    else:
        spec["kind"] = "category"
        labels = series.astype("string")
        unique = labels.dropna().unique().tolist()
        if len(unique) > 2000:
            raise ValueError(f"カテゴリーが多すぎます。ID等でないか確認してください: {column} ({len(unique)})")
        mapping = {value: f"label_{j:05d}" for j, value in enumerate(unique)}
        spec["decode"] = {v: k for k, v in mapping.items()}
        encoded[key] = labels.map(mapping).fillna("missing_category")
        cat_names.append(key)
    schema[column] = spec
if not num_names:
    raise ValueError("このノートブックでは少なくとも1つの可変数値列が必要です。")
# 公式実装のtarget欄に数値列1つを指定します。目的変数を条件固定せず全列を同時生成します。
target_key = num_names[0]
metadata = {"name": DATA_NAME, "task_type": "regression", "header": "infer",
    "column_names": encoded.columns.tolist(), "file_type": "csv",
    "num_col_idx": [encoded.columns.get_loc(c) for c in num_names if c != target_key],
    "cat_col_idx": [encoded.columns.get_loc(c) for c in cat_names],
    "target_col_idx": [encoded.columns.get_loc(target_key)],
    "data_path": f"data/{DATA_NAME}/{DATA_NAME}.csv",
    "test_path": f"data/{DATA_NAME}/holdout.csv", "val_path": None}
DATA_DIR = TABDIFF_REPO / "data" / DATA_NAME
DATA_DIR.mkdir(parents=True, exist_ok=False)
(TABDIFF_REPO / "data/Info").mkdir(parents=True, exist_ok=True)
encoded.iloc[train_idx].to_csv(DATA_DIR / f"{DATA_NAME}.csv", index=False)
encoded.iloc[test_idx].to_csv(DATA_DIR / "holdout.csv", index=False)
(TABDIFF_REPO / "data/Info" / f"{DATA_NAME}.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
pd.to_pickle({"schema": schema, "constants": constants, "columns": real.columns.tolist(),
              "train_idx": train_idx, "test_idx": test_idx}, OUT / "preprocessing.pkl")
display(pd.DataFrame({"列名": list(schema), "変換": [s["kind"] for s in schema.values()]}))
print("学習:",len(train_idx),"ホールドアウト:",len(test_idx),"モデル入力列数:",encoded.shape[1])

,列名,変換
0,年,numeric
1,都道府県名,category
2,発生年月日ウツタイン,datetime
3,性別,category
4,年齢,numeric
5,救急救命士乗車,category
6,医師の乗車,category
7,医師の２次救命処置,category
8,目撃,category
9,バイスタンダー種別,category


学習: 1286372 ホールドアウト: 142931 モデル入力列数: 68


In [4]:
# 公式スクリプトでNumPy配列とinfo.jsonを生成します。
run_command([TABDIFF_PYTHON, "process_dataset.py", "--dataname", DATA_NAME], "preprocess.log")

実行: /Users/t_aokage/Projects/Utstein_Python/.venv/bin/python process_dataset.py --dataname utstein_20261020_170859_713239
utstein_20261020_170859_713239 (1286372, 68) (0, 68) (142931, 68) (1286372, 68)
Numerical (1286372, 18)
Categorical (1286372, 49)
Processing and Saving utstein_20261020_170859_713239 Successfully!
utstein_20261020_170859_713239
Total 1429303
Train 1286372
Val 0
Test 142931
Num 19
Int 17
Cat 49


## 学習・生成の実行
公式モデル・学習器を使い、専用ラッパーで学習設定を変更します。公式側の`steps`は実際にはepoch数です。
公式実装はbestチェックポイントを4000 epoch以降に保存するため、このノートブックでは学習終了時のEMA重みを別途保存します。最良モデル選択ではありません。
学習中の大量の自動生成評価と、生成後のMLE/C2STは実行せず、最後に基本比較を行います。

In [5]:
# ラッパーは実行ごとの出力フォルダに作成し、公式ソースを書き換えません。
runner_source = r"""
import os, sys, json, argparse, importlib
from pathlib import Path
import torch
cfg = json.loads(Path(sys.argv[1]).read_text())
sys.path.insert(0, cfg['repo'])
os.chdir(cfg['repo'])
module = importlib.import_module('tabdiff.main')
Trainer = module.Trainer
original_load = module.src.load_config
original_loop = Trainer.run_loop

def configured_load(path):
    value = original_load(path)
    value['train']['main'].update(steps=cfg['epochs'], batch_size=cfg['batch'],
                                check_val_every=cfg['epochs'] + 1)
    value['sample']['batch_size'] = cfg['sample_batch']
    return value
module.src.load_config = configured_load

def train_and_save(self):
    original_loop(self)
    target = Path(self.model_save_path) / ('final_ema_%d.pt' % self.curr_epoch)
    torch.save({'denoise_fn': self.ema_model.state_dict(),
                'num_schedule': self.ema_num_schedule.state_dict(),
                'cat_schedule': self.ema_cat_schedule.state_dict()}, target)
    Path(cfg['checkpoint_manifest']).write_text(str(target.resolve()))
    print('Final EMA checkpoint:', target)
Trainer.run_loop = train_and_save

def generate_only(self):
    self.diffusion.eval()
    with torch.no_grad():
        frame = self.sample_synthetic(cfg['n_generate'], keep_nan_samples=False)
    if len(frame) != cfg['n_generate']:
        raise RuntimeError('Unexpected generated row count')
    frame.to_csv(cfg['generated_csv'], index=False)
Trainer.test = generate_only
mode = sys.argv[2]
if cfg['gpu'] >= 0 and not torch.cuda.is_available():
    raise RuntimeError('CUDA unavailable. Set GPU=-1 for CPU, or fix CUDA environment.')
args = argparse.Namespace(dataname=cfg['dataname'], mode=mode, method='tabdiff',
    gpu=cfg['gpu'], device=('cuda:%d'%cfg['gpu']) if cfg['gpu']>=0 else 'cpu',
    debug=False, no_wandb=True, exp_name=cfg['exp_name'], deterministic=True,
    y_only=False, non_learnable_schedule=False, num_samples_to_generate=cfg['n_generate'],
    ckpt_path=None, report=False, num_runs=1, impute=False)
if mode == 'test':
    args.ckpt_path = Path(cfg['checkpoint_manifest']).read_text().strip()
module.main(args)
"""
RUNNER = OUT / "run_tabdiff.py"
RUNNER.write_text(runner_source, encoding="utf-8")
config = {"repo": str(TABDIFF_REPO), "dataname": DATA_NAME, "exp_name": EXP_NAME,
          "epochs": EPOCHS, "batch": TRAIN_BATCH_SIZE, "sample_batch": SAMPLE_BATCH_SIZE,
          "gpu": GPU, "n_generate": N_GENERATE, "commit": commit,
          "checkpoint_manifest": str(OUT / "checkpoint.txt"),
          "generated_csv": str(OUT / "generated_encoded.csv")}
CONFIG = OUT / "run_config.json"
CONFIG.write_text(json.dumps(config, indent=2), encoding="utf-8")

554

In [6]:
# 学習を実行します。大規模データでは長時間かかります。
run_command([TABDIFF_PYTHON, RUNNER, CONFIG, "train"], "train.log")

実行: /Users/t_aokage/Projects/Utstein_Python/.venv/bin/python /Users/t_aokage/Projects/Utstein_Python/artifacts/18_tabdiff/20261020_170859_713239/run_tabdiff.py /Users/t_aokage/Projects/Utstein_Python/artifacts/18_tabdiff/20261020_170859_713239/run_config.json train
/Users/t_aokage/Projects/Utstein_Python/.venv/lib/python3.14/site-packages/sdmetrics/reports/single_table/__init__.py:12: FutureWarning: The single table quality report is deprecated. Please use the QualityReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.quality_report import QualityReport
/Users/t_aokage/Projects/Utstein_Python/.venv/lib/python3.14/site-packages/sdmetrics/reports/single_table/__init__.py:17: FutureWarning: The single table diagnostic report is deprecated. Please use the DiagnosticReport from 'sdmetrics.reports' instead.
  from sdmetrics.reports.single_table.diagnostic_report import DiagnosticReport
/Users/t_aokage/Projects/Utstein_Python/external/TabDiff/tabdiff/models/noise_sc

RuntimeError: 処理失敗。ログを確認: /Users/t_aokage/Projects/Utstein_Python/artifacts/18_tabdiff/20261020_170859_713239/train.log

In [ ]:
# 保存したEMAモデルから10,000例を生成します。
run_command([TABDIFF_PYTHON, RUNNER, CONFIG, "test"], "generate.log")

In [ ]:
# 内部コードを日本語列名・カテゴリー・日時・欠損へ復元します。
generated = pd.read_csv(OUT / "generated_encoded.csv", keep_default_na=False)
if len(generated) != N_GENERATE:
    raise ValueError(f"生成件数が不一致: {len(generated)}")
df_synthetic = pd.DataFrame(index=range(N_GENERATE))
for column, spec in schema.items():
    values = generated[spec["key"]]
    if spec["kind"] == "category":
        values = values.astype(str)
        unknown = ~values.isin(list(spec["decode"]) + ["missing_category"])
        if unknown.any():
            raise ValueError(f"未知の生成カテゴリー: {column}")
        decoded = values.map(spec["decode"])
        dtype = spec["dtype"]
        if isinstance(dtype, pd.CategoricalDtype):
            # 入力カテゴリ値が数値の場合も元の型へ戻す。
            reverse = {str(v): v for v in dtype.categories}
            decoded = decoded.map(reverse)
            df_synthetic[column] = pd.Categorical(decoded, dtype=dtype)
        else:
            df_synthetic[column] = decoded.astype("string")
    else:
        values = pd.to_numeric(values, errors="raise").astype(float)
        if not np.isfinite(values).all():
            raise ValueError(f"非有限の生成値: {column}")
        values = values.clip(spec["minimum"], spec["maximum"])
        if spec["integer"]:
            values = values.round()
        if "mask" in spec:
            values = values.mask(generated[spec["mask"]].eq("missing_yes"))
        if spec["kind"] == "datetime":
            df_synthetic[column] = pd.Timestamp("1970-01-01") + pd.to_timedelta(values, unit="D")
        elif spec["integer"]:
            df_synthetic[column] = values.astype("Int64")
        else:
            df_synthetic[column] = values
for column, value in constants.items():
    df_synthetic[column] = value
    if isinstance(real[column].dtype, pd.CategoricalDtype):
        df_synthetic[column] = pd.Categorical(df_synthetic[column], dtype=real[column].dtype)
df_synthetic = df_synthetic[real.columns]
# 日付と年は独立生成の不一致を残さず、生成した発生日を基準に再構成します。
if "発生年月日ウツタイン" in df_synthetic:
    event = pd.to_datetime(df_synthetic["発生年月日ウツタイン"], errors="coerce").dt.normalize()
    df_synthetic["発生年月日ウツタイン"] = event
    if "年" in df_synthetic:
        df_synthetic["年"] = event.dt.year.astype("Int64")
    if "覚知ウツタイン" in df_synthetic:
        call = pd.to_datetime(df_synthetic["覚知ウツタイン"], errors="coerce")
        df_synthetic["覚知ウツタイン"] = event + (call - call.dt.normalize()).dt.round("min")
# 01の時間採用基準を生成結果にも適用します。
for column in df_synthetic:
    if column in ["覚知現着分", "現着接触分", "接触病着分"]:
        df_synthetic[column] = df_synthetic[column].where(df_synthetic[column].between(0,120))
    elif column.startswith("覚知") and column.endswith("分"):
        low, high = (-120,0) if column == "覚知目撃分" else (-120,120)
        df_synthetic[column] = df_synthetic[column].where(df_synthetic[column].between(low,high))
assert len(df_synthetic) == 10_000
# 生成直後の内部表はgenerated_encoded.csvとして別に保持しています。
df_synthetic.to_pickle(OUT / "utstein_synthetic_10000.pkl")
df_synthetic.to_csv(OUT / "utstein_synthetic_10000.csv", index=False, encoding="utf-8-sig")
print("保存先:", OUT)
display(df_synthetic.head(20).T)

In [ ]:
# 基本比較：欠損率、数値要約、主要カテゴリー比率。不一致は勝手に修正しません。
missing_report = pd.DataFrame({"実データ欠損率": real.isna().mean(),
                              "生成データ欠損率": df_synthetic.isna().mean()})
missing_report.to_csv(OUT / "missingness_comparison.csv", encoding="utf-8-sig")
display(missing_report)
for column in ["年", "年齢", "覚知現着分", "現着接触分", "接触病着分"]:
    if column in real and pd.api.types.is_numeric_dtype(real[column]):
        display(pd.DataFrame({"実データ": real[column].describe(), "生成データ": df_synthetic[column].describe()}))
for column in ["性別", "都道府県名", "1ヶ月生存"]:
    if column in real:
        display(pd.concat([real[column].value_counts(normalize=True).rename("実データ"),
                           df_synthetic[column].value_counts(normalize=True).rename("生成データ")],axis=1).fillna(0))
issues = {}
if {"除細動","施行回数"}.issubset(df_synthetic):
    issues["除細動なし・施行回数1以上"] = int((df_synthetic["除細動"].eq("なし") & pd.to_numeric(df_synthetic["施行回数"],errors="coerce").ge(1)).sum())
if {"バイスタンダーCPR","心臓マッサージ"}.issubset(df_synthetic):
    issues["バイスタンダーCPRなし・心臓マッサージあり"] = int((df_synthetic["バイスタンダーCPR"].eq("なし") & df_synthetic["心臓マッサージ"].eq("チェックあり")).sum())
issues_report = pd.Series(issues,dtype="int64",name="生成データ内の件数").to_frame()
display(issues_report)
issues_report.to_csv(OUT / "synthetic_inconsistencies.csv",encoding="utf-8-sig")

## 実行上の補足
- CUDAのないMacでは`GPU=-1`でCPU実行できますが、全件学習は時間がかかります。UbuntuのNVIDIA GPU環境での実行を想定しています。自動でGPU2枚を並列使用する構成ではありません。
- 生成PKLは`df_synthetic`として再読込できます。入力の`utstein_modified.pkl`は上書きしません。
- 日付の年・覚知日の再構成、整数丸め、学習範囲へのクリップは後処理です。補正前の生成表も保持します。
- 基本比較は正式な精度・プライバシー評価の代わりにはなりません。ホールドアウトによる解析性能比較、近傍距離、記憶再現などは別途評価してください。
- この納品時には実データ学習・GPU動作は未実行です。コード構文と模擬データで前処理・復元を確認しています。